# Обрезка праймеров и синхронизация — человек (`PRJEB30386`)

Встроены праймер-специфические лидерные V- и константной области последовательности из Supplemental Table 2 источника датасета (`docs/datasets/PRJEB30386/technical_sequences.json`, DOI `10.3389/fimmu.2019.00660`). `MaskPrimers.py align --mode cut` обрабатывает R1 и R2; для R2 извлекается 20-nt UMI в поле `BARCODE`. Затем `PairSeq.py --coord sra` синхронизирует пары по SRA-координате и копирует `BARCODE` из R2 в R1. Стадия `pr_trimmed` публикуется атомарно.

In [ ]:
import os, sys, sysconfig, shutil, subprocess, time
from pathlib import Path

_ENV_CANDIDATES = ["/opt/conda/envs/bcr_env", "/Users/epishkin/mamba/envs/bcr_env"]
BCR_ENV = next((Path(p) for p in _ENV_CANDIDATES if (Path(p) / "bin").is_dir()), None)
if BCR_ENV is None:
    raise FileNotFoundError(f"Среда bcr_env не найдена: {_ENV_CANDIDATES}")
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
for _site in sorted((BCR_ENV / "lib").glob("python*/site-packages")):
    if str(_site) not in sys.path:
        sys.path.insert(0, str(_site))
print("BCR_ENV:", BCR_ENV)
DATASET = "PRJEB30386"
LOCAL_REPO = Path("/Users/epishkin/workspace/bcr-assembler")
VOLUME_ROOT = Path("/data/user/epishkin")

def _has_fastq(path, pattern="*.fastq.gz"):
    return Path(path).is_dir() and any(Path(path).glob(pattern))

def resolve_raw_and_result():
    remote_raw = VOLUME_ROOT / "raw" / DATASET
    local_result = LOCAL_REPO / "results" / DATASET
    local_raw = LOCAL_REPO / "raw" / DATASET
    if _has_fastq(remote_raw, "*_1.fastq.gz"):
        return remote_raw, VOLUME_ROOT / "results" / DATASET
    if _has_fastq(local_raw, "*_1.fastq.gz"):
        return local_raw, local_result
    raise FileNotFoundError(f"Парные FASTQ не найдены в {remote_raw} или {local_raw}")

def resolve_result():
    remote = VOLUME_ROOT / "results" / DATASET
    local = LOCAL_REPO / "results" / DATASET
    return remote if remote.is_dir() else local


In [ ]:
V_LEADER_PRIMERS = {
    "Hu_VH_MTPX_1": "GGTGGCAGCAGTCACAGATGCCTACTC",
    "Hu_VH_MTPX_2": "GGTGGCAGCAGCCACAGGTGCCCACTC",
    "Hu_VH_MTPX_3": "GGTGGCAGCAGCTACAGGTGTCCAGTC",
    "Hu_VH_MTPX_4": "GGTGGSAGCAGCAACARGWGCCCACTC",
    "Hu_VH_MTPX_5": "GCTGGCTGTAGCTCCAGGTGCTCACTC",
    "Hu_VH_MTPX_6": "CCTGCTGCTGACCAYCCCTTCMTGGGTCTTGTC",
    "Hu_VH_MTPX_7": "CCTGCTACTGACTGTCCCGTCCTGGGTCTTATC",
    "Hu_VH_MTPX_8": "GGGTTTTCCTCGTTGCTCTTTTAAGAGGTGTCCAGTG",
    "Hu_VH_MTPX_9": "GGGTTTTCCTTGTTGCTATTTTAAAAGGTGTCCARTG",
    "Hu_VH_MTPX_10": "GGATTTTCCTTGCTGCTATTTTAAAAGGTGTCCAGTG",
    "Hu_VH_MTPX_11": "GGGTTTTCCTTKTKGCTATWTTAGAAGGTGTCCAGTG",
    "Hu_VH_MTPX_12": "GGTGGCRGCTCCCAGATGGGTCCTGTC",
    "Hu_VH_MTPX_13": "CTGGCTGTTCTCCAAGGAGTCTGTG",
    "Hu_VH_MTPX_14": "GGCCTCCCATGGGGTGTCCTGTC",
    "Hu_VH_MTPX_15": "GGTGGCAGCAGCAACAGGTGCCCACT",
    "Hu_VH_MTPX_16": "ATGGAACTGGGGCTCCGCTGGGTTTTCC",
    "Hu_VH_MTPX_17": "ATGGACTGCACCTGGAGGATCCTCCTC",
    "Hu_VH_MTPX_18": "TGGCTGAGCTGGGTTTYCCTTGTTGC",
    "Hu_VH_MTPX_19": "GGAGTTKGGGCTGMGCTGGGTTTTCC",
    "Hu_VH_MTPX_20": "GCACCTGTGGTTTTTCCTCCTGCTGGTG",
    "Hu_VH_MTPX_21": "CACCTGTGGTTCTTCCTCCTSCTGG",
    "Hu_VH_MTPX_22": "CCAGGATGGGGTCAACCGCCATCCTC",
    "Hu_VH_MTPX_23": "CAGAGGACTCACCATGGAGTTTGGGCTGAG",
    "Hu_VH_MTPX_24": "GGACTCACCATGGAGTTGGGACTGAGC",
    "Hu_VH_MTPX_25": "GGGCTGAGCTGGCTTTTTCTTGTGGC",
    "Hu_VK_MTPX_1": "ATGTTGCCATCACAACTCATTGGGTTTCTG",
    "Hu_VK_MTPX_2": "ATGGAARCCCCAGCGCAGCTTCTCTTCC",
    "Hu_VK_MTPX_3": "ATGAGGCTCCCTGCTCAGCTCTTGGGGCT",
    "Hu_VK_MTPX_4": "ATGAGGCTCCCTGCTCAGCTCCTGGGGCT",
    "Hu_VK_MTPX_5": "ATGGACATGAGGGTCCCTGCTCAGC",
    "Hu_VK_MTPX_6": "ATGGACATGAGRGTCCTCGCTCAGC",
    "Hu_VK_MTPX_7": "ATGGAAGCCCCAGCACAGCTTCTTCTTCC",
    "Hu_VK_MTPX_8": "ATGAGGCTCCTTGCTCAGCTTCTGGGGCT",
    "Hu_VK_MTPX_9": "ATGGAAGCCCCAGCTCAGCTTCTCTTCC",
    "Hu_VK_MTPX_10": "ATGGACATGAGGGTCCCCGCTCAGC",
    "Hu_VK_MTPX_11": "ATGGGGTCCCAGGTTCACCTCCTCAG",
    "Hu_VK_MTPX_12": "ATGGTGTTGCAGACCCAGGTCTTCATTTC",
    "Hu_VK_MTPX_13": "ATGGACATGAGGGTGCCCGCTCAGC",
    "Hu_VK_MTPX_14": "CAGGAAGATGTYGCCATCACAACTCATTGG",
    "Hu_VK_MTPX_15": "CTCRCAATGAGGCTCCCTGCTCAGCTC",
    "Hu_VK_MTPX_16": "CCTGCTCAGCTCYTGGGGCTGCTAATGC",
    "Hu_VK_MTPX_17": "ATGGACATGAGGGTGCCCGCTCAGCGCC",
    "Hu_VK_MTPX_18": "ATGGACATGAGGGTSCCYGCTCAGCKCC",
    "Hu_VK_MTPX_19": "GCTCCTGGGGCTGCTAATGCTCTGG",
    "Hu_VK_MTPX_20": "GGGGCTCCTGCTGCTCTGGCTCC",
    "Hu_VK_MTPX_21": "GGACATGAGGGTCCCCGCTCAGCTCC",
    "Hu_VL_MTPX_1": "ATGGCCTGGGCTCCACTACTTCTCACCCTCC",
    "Hu_VL_MTPX_2": "ATGGCCTGGTCCCCTCTCTTCCTCACCCT",
    "Hu_VL_MTPX_3": "ATGGCCTGGGCTCTGCTCCTCCTCACCCT",
    "Hu_VL_MTPX_4": "ATGGCCTGGAYCCCTCTCCTGCTCCCCCTC",
    "Hu_VL_MTPX_5": "ATGGCCTGGGCTCTGCTGCTCCTCACTCT",
    "Hu_VL_MTPX_6": "ATGGCATGGATCCCTCTCTTCCTCGGCGTC",
    "Hu_VL_MTPX_7": "ATGGCATGGGCCACACTCCTGCTCCCACTC",
    "Hu_VL_MTPX_8": "ATGGCCTGGGTCTCCTTCTACCTACTGCCCT",
    "Hu_VL_MTPX_9": "ATGGCCTGGACTCCTCTTCTTCTCTTGCTCCT",
    "Hu_VL_MTPX_10": "ATGGCCTGGACTCCTCTCCTCCTCCTGYTCC",
    "Hu_VL_MTPX_11": "ATGAGTGTCCCCACCATGGCCTGGATGATGC",
    "Hu_VL_MTPX_12": "ATGGCCTGGGCTCCTCTGCTCCTCACCCTCC",
    "Hu_VL_MTPX_13": "ATGRCCDGCTTCCCTCTCCTCCTCACCCT",
    "Hu_VL_MTPX_14": "ATGGCCTGGACCCCACTCCTCCTCCTCTTCC",
    "Hu_VL_MTPX_15": "ATGGCCTGGGCTCTGCTSCTCCTCASCCT",
    "Hu_VL_MTPX_16": "ATGGCCTGGATCCCTCTACTTCTCCCCCTC",
    "Hu_VL_MTPX_17": "ATGGCCTGGACCSCTCTCCTCCTCRGCCTC",
    "Hu_VL_MTPX_18": "ATGGCCTGGACTCTTCTCCTTCTCGTGCTCC",
    "Hu_VL_MTPX_19": "ATGGCCTGGTCTCCTCTCCTCCTCACTCT",
    "Hu_VL_MTPX_20": "ATGCCCTGGGCTCTGCTCCTCCTGACCCT",
    "Hu_VL_MTPX_21": "ATGGCCTGGACCCCTCTCTGGCTCACTCTC",
    "Hu_VL_MTPX_22": "ATGGCCTGGACCGCTCTCCTTCTGAGCCTC",
    "Hu_VL_MTPX_23": "ATGGCTTGGACCCCACTCCTCTTCCTCACC",
    "Hu_VL_MTPX_24": "ATGGCCTGGACTCCTCTCTTTCTGTTCCTCC",
    "Hu_VL_MTPX_25": "ATGGCCTGGACTCTTCTCCTTCTCGTG",
    "Hu_VL_MTPX_26": "ATGGCCTGGACTCCTCTYCTYCTCYTG",
    "Hu_VL_MTPX_27": "ATGGCCTGGACCCCACTCCTCCTC",
    "Hu_VL_MTPX_28": "ATGGCCTGGGTCTCCTTCTACCTACTGC",
    "Hu_VL_MTPX_29": "GCAGCATCGGAGGTGCCTCAGCCATG",
    "Hu_VL_MTPX_30": "GGCAGAACTCTGGGTGTCTCACCATG",
    "Hu_VL_MTPX_31": "GCAGCACTGGTGGTGCCTCAGCCATG",
    "Hu_VL_MTPX_32": "GGGCTCTGCTSCTCCTCACYCTCCT",
    "Hu_VL_MTPX_33": "GGGCTCTGCTCCTCCTGACCCTC"
}

CONSTANT_PRIMERS = {
    "Hu_IgM_MTPX": "NHCCGACGGGGAATTCTCACAGGAGACGAGGGGGAAAAG",
    "Hu_IgM_MTPX-50": "GAGTTGTTCTTGTATTTCCAGGAGAAAGTGATGGA",
    "Hu_IgM_MTPX-100": "CCCTCTCAGGACTGATGGGAAGC",
    "Hu_IgM_MTPX-150": "CGTCCTTGGAAGGCAGCAGCACC",
    "Hu_IgG_MTPX": "GCCAGGGGGAAGACCGATGGG",
    "Hu_IgK_MTPX": "NGGGATAGAAGTTATTCAGCAGGCACACAACAGAG",
    "Hu_IgL_MTPX": "TGGCTTGRAGCTCCTCAGAGGAGG"
}

RUN_CONFIG = {
    "ERR3004229": {"v_prefix": "Hu_VH_MTPX_", "constant": ["Hu_IgM_MTPX", "Hu_IgM_MTPX-50", "Hu_IgM_MTPX-100", "Hu_IgM_MTPX-150"]},
    "ERR3004230": {"v_prefix": "Hu_VH_MTPX_", "constant": ["Hu_IgG_MTPX"]},
    "ERR3004231": {"v_prefix": "Hu_VK_MTPX_", "constant": ["Hu_IgK_MTPX"]},
    "ERR3004232": {"v_prefix": "Hu_VL_MTPX_", "constant": ["Hu_IgL_MTPX"]},
}

# В R2 перед праймером находится 20-нуклеотидный UMI.
UMI_LENGTH = 20
MAX_ERROR = 0.2
MAX_SCAN_LENGTH = 80
NPROC = 4
FORCE = False
print(f"V-leader: {len(V_LEADER_PRIMERS)}; constant: {len(CONSTANT_PRIMERS)}")


In [ ]:
def _tool(name):
    path = shutil.which(name)
    if not path:
        raise FileNotFoundError(f"Не найден executable: {name} (BCR_ENV={BCR_ENV})")
    return path

def _run_visible(cmd, stdout_log, stderr_log, outputs=(), heartbeat=30):
    stdout_log, stderr_log = Path(stdout_log), Path(stderr_log)
    stdout_log.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print("[run]", " ".join(map(str, cmd)), flush=True)
    with stdout_log.open("w") as stdout, stderr_log.open("w") as stderr:
        proc = subprocess.Popen([str(x) for x in cmd], stdout=stdout, stderr=stderr, text=True)
        print(f"PID={proc.pid}", flush=True)
        while proc.poll() is None:
            sizes = " ".join(
                f"{Path(x).name}={Path(x).stat().st_size / 1e6:.1f}MB"
                for x in outputs if Path(x).exists()
            )
            print(f"PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}", flush=True)
            time.sleep(heartbeat)
    if proc.returncode:
        raise RuntimeError(f"rc={proc.returncode}; см. {stderr_log}")

def _promote(staging, final):
    staging, final = Path(staging), Path(final)
    previous = final.parent / f".{final.name}.previous"
    if previous.exists():
        shutil.rmtree(previous)
    if final.exists():
        final.rename(previous)
    try:
        staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists():
            previous.rename(final)
        raise
    if previous.exists():
        shutil.rmtree(previous)

def _write_fasta(records, path):
    path = Path(path)
    with path.open("w") as handle:
        for name, sequence in records.items():
            handle.write(f">{name}\n{sequence}\n")
    if not path.read_text().startswith(">"):
        raise RuntimeError(f"Некорректный FASTA: {path}")
    return path

def _pick_pass(directory, prefix, marker):
    hits = sorted(Path(directory).glob(f"{prefix}*{marker}*.fastq*"))
    if len(hits) != 1:
        raise RuntimeError(f"Ожидался один {marker} для {prefix}, найдено: {hits}")
    return hits[0]

def _fastq_records(path):
    import gzip
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as handle:
        lines = sum(1 for _ in handle)
    if lines % 4:
        raise RuntimeError(f"Повреждён FASTQ: {path}")
    return lines // 4

def run_primer_trim(force=FORCE):
    result = resolve_result()
    src = result / "trimmed" / "fastq"
    final, staging = result / "pr_trimmed", result / ".pr_trimmed.staging"
    samples = sorted(p.name.removesuffix("_1.trim.fastq.gz") for p in src.glob("*_1.trim.fastq.gz"))
    if not samples or any(not (src / f"{s}_2.trim.fastq.gz").is_file() for s in samples):
        raise RuntimeError(f"Неполный набор trimmed-пар: {src}")
    if staging.exists():
        if not force:
            raise FileExistsError(f"Остался staging: {staging}; установите FORCE=True для очистки")
        shutil.rmtree(staging)
    out, logs, work, refs = (staging / x for x in ("fastq", "logs", "work", "primer_refs"))
    for d in (out, logs, work, refs):
        d.mkdir(parents=True, exist_ok=True)
    unknown = sorted(set(samples) - set(RUN_CONFIG))
    if unknown:
        raise RuntimeError(f"Нет chain-specific primer mapping для: {unknown}")
    for sample in samples:
        r1, r2 = src / f"{sample}_1.trim.fastq.gz", src / f"{sample}_2.trim.fastq.gz"
        sample_work, sample_refs = work / sample, refs / sample
        sample_work.mkdir(); sample_refs.mkdir()
        cfg = RUN_CONFIG[sample]
        v_records = {name: seq for name, seq in V_LEADER_PRIMERS.items() if name.startswith(cfg["v_prefix"])}
        c_records = {name: CONSTANT_PRIMERS[name] for name in cfg["constant"]}
        if not v_records or not c_records:
            raise RuntimeError(f"Пустой набор праймеров для {sample}")
        v_fasta = _write_fasta(v_records, sample_refs / "v_leader.fasta")
        c_fasta = _write_fasta(c_records, sample_refs / "constant.fasta")
        v_name, c_name = f"{sample}_1.v", f"{sample}_2.c"
        r1_cmd = [_tool("MaskPrimers.py"), "align", "-s", r1, "-p", v_fasta,
                  "--mode", "cut", "--maxerror", str(MAX_ERROR), "--maxlen", str(MAX_SCAN_LENGTH),
                  "--nproc", str(NPROC), "--gzip-output", "--outdir", sample_work, "--outname", v_name]
        r2_cmd = [_tool("MaskPrimers.py"), "align", "-s", r2, "-p", c_fasta,
                  "--mode", "cut", "--maxerror", str(MAX_ERROR), "--maxlen", str(MAX_SCAN_LENGTH),
                  "--barcode", "--barcodelen", str(UMI_LENGTH), "--nproc", str(NPROC),
                  "--gzip-output", "--outdir", sample_work, "--outname", c_name]
        _run_visible(r1_cmd, logs / f"{sample}.R1.mask.stdout.log", logs / f"{sample}.R1.mask.stderr.log")
        _run_visible(r2_cmd, logs / f"{sample}.R2.mask.stdout.log", logs / f"{sample}.R2.mask.stderr.log")
        r1_pass = _pick_pass(sample_work, v_name, "primers-pass")
        r2_pass = _pick_pass(sample_work, c_name, "primers-pass")
        sync_dir = sample_work / "sync"
        sync_dir.mkdir()
        pair_cmd = [_tool("PairSeq.py"), "-1", r1_pass, "-2", r2_pass, "--coord", "sra",
                    "--2f", "BARCODE", "--gzip-output", "--outdir", sync_dir, "--outname", sample]
        _run_visible(pair_cmd, logs / f"{sample}.pairseq.stdout.log", logs / f"{sample}.pairseq.stderr.log")
        p1 = _pick_pass(sync_dir, sample, "-1_pair-pass")
        p2 = _pick_pass(sync_dir, sample, "-2_pair-pass")
        n1, n2 = _fastq_records(p1), _fastq_records(p2)
        if n1 == 0 or n1 != n2:
            raise RuntimeError(f"PairSeq {sample}: R1={n1}, R2={n2}")
        p1.rename(out / f"{sample}_1.pr.fastq.gz")
        p2.rename(out / f"{sample}_2.pr.fastq.gz")
    expected = {f"{s}_{mate}.pr.fastq.gz" for s in samples for mate in (1, 2)}
    if {p.name for p in out.glob("*.pr.fastq.gz")} != expected:
        raise RuntimeError("Проверка комплекта pr_trimmed FASTQ не пройдена")
    shutil.rmtree(work); shutil.rmtree(refs)
    _promote(staging, final)
    print("Готово:", final)


## Запуск

`FORCE=True` разрешает удалить только оставшийся промежуточный каталог; существующая каноническая стадия сохраняется до успешной проверки нового результата.

In [ ]:
run_primer_trim(force=FORCE)
